In [1]:
import sqlite3
import datetime

db_name="game_usage.db"
connection=sqlite3.connect(db_name)

cursor=connection.cursor()
time_data=datetime.datetime.now()
date=time_data.date()

game_name="Tavern.exe"


In [2]:
def handel_sqlite_exception(func):
    def wrapper(*args,**kwargs):
        try:
            result =func(*args,**kwargs)
            print(result)
            return result
        except sqlite3.Error as e:
            print(e)
    return wrapper

In [3]:
games_table_name="GAMES_DATA"
create_table=f"""CREATE TABLE IF NOT EXISTS {games_table_name}(day STRING,
                                                game_name STRING,
                                                date  STRING PRIMARY KEY, 
                                                total_time_played INTEGER,  
                                                last_played STRING );"""
cursor.execute(create_table)
connection.commit()

In [4]:
@handel_sqlite_exception
def update_game_data(date,table_name,total_usage,last_played):
    
        
        update_query = f"""
                UPDATE {table_name}
                SET total_time_played = ?,
                        last_played = ?
                WHERE date = ? AND game_name = ?
                """
        result = cursor.execute(update_query,(total_usage,
                                    last_played,
                                    date,
                                    game_name))
        connection.commit()
        return result 
        

In [20]:
#Debug Testing
print(update_game_data(date=date,
                       table_name=games_table_name,
                       total_usage=190,
                       last_played=f"{time_data.hour}:{time_data.minute}"))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14648\3634383740.py:11: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  result = cursor.execute(update_query,(total_usage,


In [6]:
@handel_sqlite_exception
def does_record_exits(date,game_name):
    global games_table_name
    serach_query=f"""SELECT * FROM {games_table_name} WHERE game_name = ? AND date= ?"""
    cursor.execute(serach_query,(game_name,date))
    result=cursor.fetchone()
    if result is not None:
        return True
    return False

In [7]:

print("game found",does_record_exits(date=date,game_name=game_name))

True
game found True


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14648\432336041.py:5: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute(serach_query,(game_name,date))


In [8]:
#Working
@handel_sqlite_exception
def create_game_record(day,game_name,date,time,total_time_played=0):
   
        
        create_query=f"""INSERT INTO {games_table_name} (day,
                                                game_name,
                                                date,
                                                total_time_played,
                                                last_played) 
                                                VALUES (?,?,?,?,?)"""

        
        cursor.execute(create_query,(day,game_name,date,total_time_played,time))
        
        connection.commit()
        return True

 

In [9]:

print(create_game_record(day=str(time_data.strftime("%A")),
                         game_name="DarkAndDarker",
                         date=str(time_data.date()),
                         time=(time_data.hour*60+time_data.minute),
                         total_time_played=60))


UNIQUE constraint failed: GAMES_DATA.date
None


In [10]:
print(type(time_data.day),time_data.day)
print(time_data.strftime("%A"))

<class 'int'> 18
Tuesday


In [11]:
cursor.execute(f"""SELECT * FROM {games_table_name}""")
result=cursor.fetchall()
print(result)

[('Saturday', 'Tavern.exe', '2026-08-15', 120, '20:39'), ('Tuesday', 'Tavern.exe', '2026-08-18', 120, '16:39')]


In [12]:
#cursor.execute(f"""DELETE FROM {games_table_name}""")
#connection.commit()

In [13]:
#cursor.execute(f"DELETE FROM {games_table_name}")
connection.commit()

In [14]:
table_name="game_time_limits"
delete_table=f"DROP TABLE {table_name}"
create_table=f"""CREATE TABLE IF NOT EXISTS {table_name} (game_name STRING UNIQUE,
                                                    time_limit INTEGER)"""
insert_table=f"""INSERT INTO {table_name} (game_name,
                                            time_limit)
                                            VALUES
                                            (?,?)"""
update_table=f"""UPDATE {table_name} SET time_limit = 60  WHERE game_name = ?"""
read_table=f"SELECT * FROM {table_name}"
game_1=("Tavern.exe",120)
cursor.execute(read_table)
data=cursor.fetchone()
print(data)
connection.commit()                                                   

('Tavern.exe', 120)


In [15]:
#Working
game_limits="game_time_limits"
@handel_sqlite_exception
def get_game_limit(game_name,game_limits):
    cursor.execute(f"""SELECT time_limit FROM {game_limits} WHERE game_name = ?""",(game_name,))
    result=cursor.fetchone()
    return result[0]

In [16]:
#Working
@handel_sqlite_exception
def get_total_time_played(game_name, games_table_name, date):
    cursor.execute(
        f"""SELECT total_time_played
            FROM {games_table_name}
            WHERE game_name = ? AND date = ?""",
        (game_name, date)
    )

    result = cursor.fetchone()
    return result[0] if result else 0
    

In [17]:
print(get_total_time_played(game_name="DarkAndDarker",date=date,games_table_name=games_table_name))

0
0


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14648\3549087.py:4: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute(


In [18]:
def total_time_played_today(game_name,games_table_name,date):
    cursor.execute("""SELECT total_time_played""")